# 22 – Joining Tract Geometries and ACS Attributes

This notebook joins:

- **Tract boundaries** (GeoDataFrame),
- **ACS 2019–2023 attributes** (DataFrame).

**Goals:**
- Load cleaned tract geometries and ACS attributes.
- Ensure consistent GEOID formatting.
- Join ACS data onto tract geometries.
- Save tract + ACS layer for mapping and analysis.

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CENSUS_DIR = DATA_DIR / "census" / "processed"

tracts_clean_path = CENSUS_DIR / "tracts_clean.gpkg"
acs_clean_path = CENSUS_DIR / "acs_2019_2023_tracts_clean.csv"

tracts_clean_path, acs_clean_path

(WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/processed/tracts_clean.gpkg'),
 WindowsPath('c:/Users/arjav/DevSource/toc-performance-dashboard/data/census/processed/acs_2019_2023_tracts_clean.csv'))

In [2]:
tracts = gpd.read_file(tracts_clean_path)
acs = pd.read_csv(acs_clean_path)

print("Tracts:", len(tracts))
print("ACS rows:", len(acs))

Tracts: 1009
ACS rows: 1766


In [3]:
tracts.columns

Index(['GISJOIN', 'geometry'], dtype='object')

In [4]:
acs.columns

Index(['GISJOIN', 'ASNQE001', 'ASQPE001', 'ASVBE001', 'ASS8E001', 'ASS8E002',
       'ASS8E003', 'ASS9E001', 'ASS9E002', 'ASS9E003', 'ASORE001', 'ASORE003',
       'ASORE004', 'ASORE010', 'ASORE018', 'ASORE019', 'ASORE021', 'ASORE002'],
      dtype='object')

In [5]:
TRACT_GEOID_FIELD_TRACTS = "GISJOIN"
TRACT_GEOID_FIELD_ACS = "GISJOIN"

TRACT_GEOID_FIELD_TRACTS, TRACT_GEOID_FIELD_ACS

('GISJOIN', 'GISJOIN')

In [6]:
tracts[TRACT_GEOID_FIELD_TRACTS] = (
    tracts[TRACT_GEOID_FIELD_TRACTS]
    .astype(str)
    .str.strip()
)

acs[TRACT_GEOID_FIELD_ACS] = (
    acs[TRACT_GEOID_FIELD_ACS]
    .astype(str)
    .str.strip()
)

In [7]:
tracts_acs = tracts.merge(
    acs,
    how="left",
    left_on=TRACT_GEOID_FIELD_TRACTS,
    right_on=TRACT_GEOID_FIELD_ACS,
    suffixes=("", "_acs")
)

print("Rows after join (must match tract count):", len(tracts_acs))
tracts_acs.head()

Rows after join (must match tract count): 1009


,GISJOIN,geometry,ASNQE001,ASQPE001,ASVBE001,ASS8E001,ASS8E002,ASS8E003,ASS9E001,ASS9E002,ASS9E003,ASORE001,ASORE003,ASORE004,ASORE010,ASORE018,ASORE019,ASORE021,ASORE002
0,G0400130010102,"POLYGON ((827801.708 1091626.91, 827865.552 10...",6265,188486,-666666666,3537,2824,713,2824,2731,93,2016,931,93,22,19,110,773,1024
1,G0400130010103,"POLYGON ((765755.318 1002785.693, 765592.367 1...",3681,117813,2744,1884,1516,368,1516,1451,65,1735,1188,60,0,0,0,462,1248
2,G0400130010104,"POLYGON ((765755.318 1002785.693, 765757.331 1...",3131,140587,-666666666,2447,1737,710,1737,1681,56,915,316,0,0,0,0,584,316
3,G0400130030401,"POLYGON ((707726.563 1037891.22, 707726.326 10...",4744,145865,1679,3170,2423,747,2423,2221,202,1563,596,152,0,0,59,734,748
4,G0400130030402,"POLYGON ((707726.563 1037891.22, 707707.434 10...",4153,108031,2096,2273,1928,345,1928,1803,125,1717,960,37,0,0,32,642,997


In [8]:
no_match_count = tracts_acs[TRACT_GEOID_FIELD_ACS].isna().sum()
match_rate = 1 - no_match_count / len(tracts_acs)

print("Tracts without ACS data:", no_match_count)
print("Match rate:", round(match_rate * 100, 2), "%")

Tracts without ACS data: 0
Match rate: 100.0 %


In [9]:
tracts_acs_path = CENSUS_DIR / "tracts_acs_2019_2023.gpkg"
tracts_acs.to_file(tracts_acs_path, driver="GPKG")

print("Saved tract+ACS layer to:", tracts_acs_path)

Saved tract+ACS layer to: c:\Users\arjav\DevSource\toc-performance-dashboard\data\census\processed\tracts_acs_2019_2023.gpkg
